# 01. Single trajectory synthesis and score analysis

This notebook reproduces the **single-trajectory** part of the research workflow:
1. Parse and clean saved trajectories/scores from CSV exports.
2. Find the best observed trajectory in the dataset.
3. Rebuild a best control sequence for one initial condition with simulated annealing.
4. Recalculate and compare objective values using the paper/solver Bolza functional.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

base = Path.cwd().resolve()
if (base / 'src').exists():
    ROOT = base
elif (base / 'research' / 'src').exists():
    ROOT = base / 'research'
elif (base.parent / 'src').exists():
    ROOT = base.parent
else:
    raise RuntimeError('Cannot locate research/src directory from current working directory')

SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.append(str(SRC))
from dynamics import CostConfig, build_trajectory_bundle, bolza_cost, clamp_controls, load_training_samples

plt.style.use('seaborn-v0_8-whitegrid')
np.random.seed(42)


In [ ]:
DATA_DIR = ROOT / 'src' / 'data'
samples = load_training_samples(DATA_DIR)
bundle = build_trajectory_bundle(samples)

print('Samples:', samples.shape)
print('Trajectories with scores:', len(bundle))
print('Steps range:', int(samples['step'].min()), '..', int(samples['step'].max()))
print(samples.head(3))


In [ ]:
score_by_traj = samples[['trajectory_id', 'score']].drop_duplicates().sort_values('score')
best_tid = int(score_by_traj.iloc[0]['trajectory_id'])
worst_tid = int(score_by_traj.iloc[-1]['trajectory_id'])

print('Best observed trajectory id:', best_tid)
print('Best observed score:', float(score_by_traj.iloc[0]['score']))
print('Worst observed score:', float(score_by_traj.iloc[-1]['score']))
print('
Top-10 observed trajectories:')
print(score_by_traj.head(10).to_string(index=False))


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].hist(score_by_traj['score'], bins=40, color='#3A6EA5', alpha=0.85)
ax[0].axvline(score_by_traj.iloc[0]['score'], color='crimson', linestyle='--', linewidth=2, label='best observed')
ax[0].set_title('Trajectory score distribution')
ax[0].set_xlabel('score')
ax[0].set_ylabel('count')
ax[0].legend()

ax[1].plot(score_by_traj['score'].to_numpy(), color='#1B998B', linewidth=1.8)
ax[1].set_title('Sorted scores (lower is better)')
ax[1].set_xlabel('rank')
ax[1].set_ylabel('score')

plt.tight_layout()
plt.show()


In [ ]:
cfg = CostConfig()
best_obs = bundle[best_tid]
X_obs = best_obs['X']
U_obs = best_obs['U']

calc_score = bolza_cost(X_obs[0], U_obs, cfg)
print('Best observed score from DB:', best_obs['score'])
print('Recomputed score (paper functional):', calc_score)
print('Absolute difference:', abs(best_obs['score'] - calc_score))


In [ ]:
# Simulated annealing for one selected initial state.
# We optimize controls directly and keep solver-compatible bounds.
def random_controls(n_steps: int) -> np.ndarray:
    low = np.array([-np.pi/12, -np.pi, -np.pi/12, 0.0])
    high = np.array([ np.pi/12,  np.pi,  np.pi/12, 12.0])
    return low + (high - low) * np.random.rand(n_steps, 4)


def anneal_single(initial_state, start_controls, cfg, n_iter=25000, step_size=0.06, t0=120.0):
    current_u = clamp_controls(start_controls.copy())
    current_j = bolza_cost(initial_state, current_u, cfg)
    best_u = current_u.copy()
    best_j = current_j
    trace = [current_j]

    for i in range(n_iter):
        temp = t0 / (1.0 + 0.01 * i)
        cand_u = clamp_controls(current_u + np.random.normal(scale=step_size, size=current_u.shape))
        cand_j = bolza_cost(initial_state, cand_u, cfg)

        accept = (cand_j < current_j) or (np.random.rand() < np.exp((current_j - cand_j) / max(temp, 1e-9)))
        if accept:
            current_u, current_j = cand_u, cand_j
            if cand_j < best_j:
                best_u, best_j = cand_u.copy(), cand_j

        if i % 100 == 0:
            trace.append(best_j)

    return best_u, best_j, np.array(trace)

initial_state = X_obs[0].copy()
start_u = U_obs.copy()

best_u_opt, best_j_opt, trace = anneal_single(initial_state, start_u, cfg)
print('Observed best score for selected initial state:', best_obs['score'])
print('Optimized score from annealing:', best_j_opt)
print('Improvement:', float(best_obs['score'] - best_j_opt))


In [ ]:
from dynamics import rollout

X_opt = rollout(initial_state, best_u_opt, cfg.dt)

fig, ax = plt.subplots(1, 2, figsize=(13, 5))

# x-z geometry with obstacles and window
ax[0].plot(X_obs[:, 0], X_obs[:, 2], '-o', markersize=3, linewidth=1.5, color='#6C757D', label='best observed')
ax[0].plot(X_opt[:, 0], X_opt[:, 2], '-o', markersize=3, linewidth=2.0, color='#D1495B', label='optimized single')

for cyl in cfg.cylinders:
    c = plt.Circle((cyl.x, cyl.z), cyl.radius, color='#F4A261', alpha=0.28)
    ax[0].add_patch(c)
for wnd in cfg.windows:
    w = plt.Circle((wnd.x, wnd.z), wnd.radius, color='#2A9D8F', alpha=0.35)
    ax[0].add_patch(w)

ax[0].scatter([cfg.terminal_state[0]], [cfg.terminal_state[2]], marker='*', s=160, color='black', label='terminal')
ax[0].set_title('Trajectory in x-z plane')
ax[0].set_xlabel('x1')
ax[0].set_ylabel('x3')
ax[0].axis('equal')
ax[0].legend(loc='best')

ax[1].plot(np.arange(len(trace)) * 100, trace, color='#1D3557')
ax[1].set_title('Annealing progress')
ax[1].set_xlabel('iteration')
ax[1].set_ylabel('best score')

plt.tight_layout()
plt.show()


In [ ]:
control_names = ['phi (roll)', 'theta (pitch)', 'psi (yaw)', 'thrust']
fig, axes = plt.subplots(2, 2, figsize=(12, 7), sharex=True)
axes = axes.ravel()
steps = np.arange(1, U_obs.shape[0] + 1)

for j, ax in enumerate(axes):
    ax.plot(steps, U_obs[:, j], '-o', color='#6C757D', linewidth=1.3, markersize=3, label='observed')
    ax.plot(steps, best_u_opt[:, j], '-o', color='#D1495B', linewidth=1.3, markersize=3, label='optimized')
    ax.set_title(control_names[j])
    ax.set_xlabel('step')
    ax.set_ylabel('value')

axes[0].legend()
plt.tight_layout()
plt.show()


### Result summary

- The notebook identifies the best stored single trajectory in the dataset.
- It recomputes the paper score from dynamics + penalties for consistency checks.
- It then runs trajectory-level simulated annealing from the same initial state and reports the improved score.
